In [1]:
import numpy as np
from tensorflow.keras.datasets import cifar10

# 1. Download CIFAR-10
(x_train, y_train), (x_test, y_test) = cifar10.load_data()
y_train = y_train.flatten()
y_test = y_test.flatten()

# CIFAR-10 labels:
# 0: airplane, 1: automobile, 2: bird, 3: cat, 4: deer,
# 5: dog, 6: frog, 7: horse, 8: ship, 9: truck

# 2. Convert to CIFAR-2 → animals (1) vs man-made (0)
animal_classes = {2, 3, 4, 5, 6, 7}

def to_cifar2_labels(y):
    return np.array([1 if label in animal_classes else 0 for label in y], dtype=np.uint8)

y_train_bin = to_cifar2_labels(y_train)
y_test_bin = to_cifar2_labels(y_test)

# 3. Keep original pixel values (0–255)
x_train_clean = x_train.astype(np.uint8)
x_test_clean = x_test.astype(np.uint8)

# Sanity check: Ensure shape (num_images, 32, 32, 3)
print("Train images shape:", x_train_clean.shape)
print("Test images shape:", x_test_clean.shape)

# 4. Save in .npy format — each image is a 3D array (32x32x3)
np.save("cifar_train_images.npy", x_train_clean)
np.save("cifar_train_labels.npy", y_train_bin)
np.save("cifar_test_images.npy", x_test_clean)
np.save("cifar_test_labels.npy", y_test_bin)

print("\n✅ Saved CIFAR-2 files in .npy format:")
print("- cifar_train_images.npy  (shape:", x_train_clean.shape, ")")
print("- cifar_train_labels.npy  (shape:", y_train_bin.shape, ")")
print("- cifar_test_images.npy   (shape:", x_test_clean.shape, ")")
print("- cifar_test_labels.npy   (shape:", y_test_bin.shape, ")")

I0000 00:00:1775903469.965671 3513410 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1775903470.019724 3513410 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1775903471.962528 3513410 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
/home/ubuntu/local/fractal_env/lib/python3.12/site-packages/keras/src/datasets/cifar.py:18: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  d = cPickle.load(f, encoding="bytes")


Train images shape: (50000, 32, 32, 3)
Test images shape: (10000, 32, 32, 3)

✅ Saved CIFAR-2 files in .npy format:
- cifar_train_images.npy  (shape: (50000, 32, 32, 3) )
- cifar_train_labels.npy  (shape: (50000,) )
- cifar_test_images.npy   (shape: (10000, 32, 32, 3) )
- cifar_test_labels.npy   (shape: (10000,) )


In [2]:
import os
import cv2
import ctypes
import numpy as np

# =========================
# LOAD RESIZE LIBRARY
# =========================
lib_resize = ctypes.CDLL("./libresize.so")

lib_resize.resize_image.argtypes = [
    ctypes.POINTER(ctypes.c_double),
    ctypes.c_int,
    ctypes.c_int,
    ctypes.POINTER(ctypes.c_double),
    ctypes.c_int,
    ctypes.c_int
]

lib_resize.resize_image.restype = None

# =========================
# LOAD CIFAR-2 ARRAYS
# =========================
x_train = np.load("cifar_train_images.npy")
y_train = np.load("cifar_train_labels.npy")

x_test = np.load("cifar_test_images.npy")
y_test = np.load("cifar_test_labels.npy")

print("Train:", x_train.shape)
print("Test :", x_test.shape)

# =========================
# OUTPUT DIRECTORY
# =========================
base_dir = "/home/ubuntu/cifar2_images_resized"

for split in ["train", "test"]:
    for label in ["0", "1"]:
        os.makedirs(
            os.path.join(base_dir, split, label),
            exist_ok=True
        )

# =========================
# RESIZE FUNCTION
# 32x32 RGB
#    ↓
# libresize.so
#    ↓
# 64x64 RGB
# =========================
def resize_cifar_lib(img):

    img = img.astype(np.float64)

    old_h, old_w = img.shape[:2]

    out = np.zeros((64, 64, 3), dtype=np.float64)

    lib_resize.resize_image(
        img.ravel().ctypes.data_as(ctypes.POINTER(ctypes.c_double)),
        old_h,
        old_w,
        out.ravel().ctypes.data_as(ctypes.POINTER(ctypes.c_double)),
        64,
        64
    )

    return np.clip(out, 0, 255).astype(np.uint8)

# =========================
# SAVE TRAIN IMAGES
# =========================
for idx, (img, label) in enumerate(zip(x_train, y_train)):

    img_resized = resize_cifar_lib(img)

    save_path = os.path.join(
        base_dir,
        "train",
        str(label),
        f"{idx}.png"
    )

    # RGB -> BGR for OpenCV
    img_bgr = cv2.cvtColor(
        img_resized,
        cv2.COLOR_RGB2BGR
    )

    cv2.imwrite(save_path, img_bgr)

    if (idx + 1) % 5000 == 0:
        print(f"Train: {idx+1}/{len(x_train)} saved")

# =========================
# SAVE TEST IMAGES
# =========================
for idx, (img, label) in enumerate(zip(x_test, y_test)):

    img_resized = resize_cifar_lib(img)

    save_path = os.path.join(
        base_dir,
        "test",
        str(label),
        f"{idx}.png"
    )

    img_bgr = cv2.cvtColor(
        img_resized,
        cv2.COLOR_RGB2BGR
    )

    cv2.imwrite(save_path, img_bgr)

    if (idx + 1) % 2000 == 0:
        print(f"Test: {idx+1}/{len(x_test)} saved")

print("\n✅ CIFAR-2 resized to 64×64 using libresize.so!")

# =========================
# VERIFY ONE IMAGE
# =========================
sample = cv2.imread(
    os.path.join(base_dir, "train", "0", "1.png"),
    cv2.IMREAD_UNCHANGED
)

print("\nVerification:")
print("Shape:", sample.shape)
print("Dtype:", sample.dtype)

Train: (50000, 32, 32, 3)
Test : (10000, 32, 32, 3)
Train: 5000/50000 saved
Train: 10000/50000 saved
Train: 15000/50000 saved
Train: 20000/50000 saved
Train: 25000/50000 saved
Train: 30000/50000 saved
Train: 35000/50000 saved
Train: 40000/50000 saved
Train: 45000/50000 saved
Train: 50000/50000 saved
Test: 2000/10000 saved
Test: 4000/10000 saved
Test: 6000/10000 saved
Test: 8000/10000 saved
Test: 10000/10000 saved

✅ CIFAR-2 resized to 64×64 using libresize.so!

Verification:
Shape: (64, 64, 3)
Dtype: uint8
